In [4]:
from pathlib import Path
import logging
import shutil
from datetime import datetime


def localizar_raiz_projeto():
    """Encontra a raiz do projeto ao executar da raiz ou de uma subpasta."""
    for pasta in (Path.cwd(), *Path.cwd().parents):
        if (pasta / "data").is_dir() and (pasta / "lake").is_dir():
            return pasta

    raise FileNotFoundError(
        "Nao foi possivel localizar a raiz do projeto (pastas 'data' e 'lake')."
    )


def configurar_logger(raiz_projeto, nome_arquivo="data_lake.txt"):
    pasta_logs = raiz_projeto / "logs"
    pasta_logs.mkdir(parents=True, exist_ok=True)
    caminho_arquivo = pasta_logs / nome_arquivo

    logging.basicConfig(
        filename=caminho_arquivo,
        filemode="a",
        format="%(asctime)s - %(levelname)s - %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
        level=logging.INFO,
        force=True,
    )

    return logging.getLogger(__name__)


def verificar_arquivo(caminho):
    return caminho.is_file()


def main():
    try:
        raiz_projeto = localizar_raiz_projeto()
    except FileNotFoundError as erro:
        print(erro)
        return

    logger = configurar_logger(raiz_projeto)
    caminho_csv = raiz_projeto / "data" / "ecommerce_orders_dirty.csv"

    logger.info("Aplicacao iniciada.")
    logger.info("Verificando arquivo de origem.")

    if not verificar_arquivo(caminho_csv):
        logger.error("Arquivo nao encontrado: %s", caminho_csv)
        print(f"Arquivo CSV nao encontrado: {caminho_csv}. Ingestao cancelada.")
        return

    nome_arquivo = caminho_csv.name
    data_ingestao = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    logger.info("Nome do arquivo: %s", nome_arquivo)
    logger.info("Origem: %s", caminho_csv)
    logger.info("Data da ingestao: %s", data_ingestao)

    pasta_bronze = raiz_projeto / "lake" / "bronze" / "ecommerce"
    pasta_bronze.mkdir(parents=True, exist_ok=True)
    caminho_destino = pasta_bronze / nome_arquivo

    try:
        shutil.copy2(caminho_csv, caminho_destino)
    except OSError:
        logger.exception("Falha durante a copia do arquivo.")
        print("Erro ao copiar o arquivo. Consulte o log em logs/data_lake.txt.")
        return

    logger.info("Arquivo copiado para Bronze: %s", caminho_destino)
    logger.info("Ingestao concluida com sucesso.")
    logger.info("Aplicacao finalizada.")
    print(f"Ingestao concluida! Arquivo copiado para: {caminho_destino}")


if __name__ == "__main__":
    main()


Ingestao concluida! Arquivo copiado para: c:\Users\hytal\Desktop\DataLake\lake\bronze\ecommerce\ecommerce_orders_dirty.csv
